# Fine-tuning — Epochs 2 & 3 (GPU FAISS, resumed from epoch 1 checkpoint)

This notebook:
- Loads the epoch 1 best checkpoint
- Moves the FAISS index to GPU (8.6x faster: 107ms → 12ms per query)
- Runs 2 epochs (epochs 2 and 3)

Run in a **separate kernel** — the original notebook can keep running independently.

In [1]:
%run ../setup_env.py

Checking environment...
  ✓ faiss
  ✓ datasets


2026-05-27 07:07:09.291422: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.

  ✓ evaluate
  ✓ accelerate
  ✓ sentencepiece

All packages present — ready to go
FAISS patch applied — version 1.14.1


In [2]:
import os
import sys
import json
import faiss
import torch
import numpy as np
import random
import yaml
import transformers.utils.import_utils as import_utils
import transformers.utils as tu

# faiss patch
if hasattr(import_utils.is_faiss_available, "cache_clear"):
    import_utils.is_faiss_available.cache_clear()
import_utils._faiss_available = True
import_utils.is_faiss_available = lambda: True
tu.is_faiss_available = lambda: True

# paths
REPO_ROOT      = os.path.abspath(os.path.join(os.getcwd(), "../.."))
FEVER_DIR      = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR       = os.path.join(FEVER_DIR, "data")
RESULTS_DIR    = os.path.join(FEVER_DIR, "results")
CONFIG_DIR     = os.path.join(FEVER_DIR, "configs")
CHECKPOINT_DIR = os.path.join(FEVER_DIR, "results", "checkpoints")

os.makedirs(RESULTS_DIR, exist_ok=True)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Fever dir:      {FEVER_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"FAISS patch:    {import_utils.is_faiss_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

Fever dir:      /home/jovyan/lectures/raq-reproducibility-challenge/fever
Checkpoint dir: /home/jovyan/lectures/raq-reproducibility-challenge/fever/results/checkpoints
FAISS patch:    True
GPU: NVIDIA A40
VRAM free: 38.6 GB


In [3]:
from datasets import load_dataset

with open(os.path.join(CONFIG_DIR, "fever_config.yaml"), "r") as f:
    config = yaml.safe_load(f)

LABEL2ID = {
    "SUPPORTS":        "0",
    "REFUTES":         "1",
    "NOT ENOUGH INFO": "2"
}
LABEL_TOKEN_IDS = [288, 134, 176]
MAX_LENGTH = config["training"]["max_source_length"]

print("Config loaded")
print(f"  n_docs:        {config['model']['n_docs']}")
print(f"  learning_rate: {config['training']['learning_rate']}")
print(f"  max_length:    {MAX_LENGTH}")

Config loaded
  n_docs:        5
  learning_rate: 3e-05
  max_length:    300


In [4]:
# load passages
print("Loading passages...")
passages = []
with open(os.path.join(DATA_DIR, "fever_passages.jsonl")) as f:
    for line in f:
        passages.append(json.loads(line))
print(f"Passages:  {len(passages):,}")

# load FAISS index and move to GPU
print("Loading FAISS index...")
cpu_index = faiss.read_index(
    os.path.join(DATA_DIR, "fever_faiss.index")
)
print(f"CPU index: {cpu_index.ntotal:,} vectors")

print("Moving index to GPU...")
res       = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
print(f"GPU index: {gpu_index.ntotal:,} vectors — ready")

# load dataset
print("\nLoading FEVER dataset...")
dataset = load_dataset("copenlu/fever_gold_evidence")
print(f"Train: {len(dataset['train']):,}  Val: {len(dataset['validation']):,}")

Loading passages...
Passages:  574,197
Loading FAISS index...
CPU index: 574,197 vectors
Moving index to GPU...
GPU index: 574,197 vectors — ready

Loading FEVER dataset...
Train: 228,277  Val: 15,935


In [5]:
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizerFast,
    BartForConditionalGeneration,
    BartTokenizer
)

print("Loading DPR question encoder...")
q_tokenizer = DPRQuestionEncoderTokenizerFast.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)
q_encoder = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to("cuda")
print(f"DPR on: {next(q_encoder.parameters()).device}")

print("Loading BART...")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
model = BartForConditionalGeneration.from_pretrained(
    "facebook/bart-large"
).to("cuda")
print(f"BART on: {next(model.parameters()).device}")

total = torch.cuda.get_device_properties(0).total_memory / 1024**3
used  = torch.cuda.memory_reserved() / 1024**3
print(f"VRAM: {used:.1f} GB used / {total:.1f} GB total")

Loading DPR question encoder...


Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


DPR on: cuda:0
Loading BART...
BART on: cuda:0
VRAM: 2.0 GB used / 44.4 GB total


In [7]:
# show available checkpoints
print("Available checkpoints:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    size = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1024**3
    print(f"  {f}: {size:.2f} GB")

# load epoch 1 best checkpoint
print("\nLoading best checkpoint (epoch 1)...")
q_encoder.load_state_dict(
    torch.load(
        os.path.join(CHECKPOINT_DIR, "q_encoder_epoch2_gpu.pt"),
        map_location="cuda"
    )
)
model.load_state_dict(
    torch.load(
        os.path.join(CHECKPOINT_DIR, "bart_epoch2_gpu.pt"),
        map_location="cuda"
    )
)
print("Checkpoint loaded successfully")

Available checkpoints:
  bart_best.pt: 1.51 GB
  bart_best_gpu.pt: 1.51 GB
  bart_epoch1.pt: 1.51 GB
  bart_epoch2_gpu.pt: 1.51 GB
  q_encoder_best.pt: 0.41 GB
  q_encoder_best_gpu.pt: 0.41 GB
  q_encoder_epoch1.pt: 0.41 GB
  q_encoder_epoch2_gpu.pt: 0.41 GB

Loading best checkpoint (epoch 1)...
Checkpoint loaded successfully


In [8]:
def search_index(claim, index, passages, q_encoder,
                 q_tokenizer, n_docs=config["model"]["n_docs"],
                 training=False):
    encoded = q_tokenizer(
        claim, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH
    )
    if training:
        query_vec = q_encoder(
            input_ids=encoded["input_ids"].to("cuda"),
            attention_mask=encoded["attention_mask"].to("cuda")
        ).pooler_output
        query_vec_np = query_vec.detach().cpu().numpy()
    else:
        with torch.no_grad():
            query_vec = q_encoder(
                input_ids=encoded["input_ids"].to("cuda"),
                attention_mask=encoded["attention_mask"].to("cuda")
            ).pooler_output
        query_vec_np = query_vec.cpu().numpy()

    query_vec_np_norm = query_vec_np.copy()
    faiss.normalize_L2(query_vec_np_norm)

    # always use gpu_index regardless of index argument
    scores, indices = gpu_index.search(
        query_vec_np_norm.astype("float32"), n_docs
    )
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "rank":  len(results) + 1,
            "score": float(score),
            "title": passages[idx]["title"],
            "text":  passages[idx]["text"],
            "idx":   int(idx)
        })
    return results, query_vec if training else None


def prepare_bart_inputs(claim, retrieved_passages,
                         bart_tokenizer, max_length):
    texts = [
        f"question: {claim} title: {p['title']} context: {p['text']}"
        for p in retrieved_passages
    ]
    return bart_tokenizer(
        texts, max_length=MAX_LENGTH,
        truncation=True, padding="max_length",
        return_tensors="pt"
    )


def forward_pass(claim, retrieved_results, gold_label_str,
                 model, bart_tokenizer, label2id):
    encoded = prepare_bart_inputs(
        claim, retrieved_results, bart_tokenizer, MAX_LENGTH
    )
    retrieval_scores = [r["score"] for r in retrieved_results]
    K             = len(retrieved_results)
    gold_token_id = LABEL_TOKEN_IDS[int(label2id[gold_label_str])]
    labels        = torch.full((K, 1), gold_token_id)
    output = model(
        input_ids      = encoded["input_ids"].to("cuda"),
        attention_mask = encoded["attention_mask"].to("cuda"),
        labels         = labels.to("cuda")
    )
    logits       = output.logits[:, 0, :]
    label_logits = logits[:, LABEL_TOKEN_IDS]
    retrieval_probs = torch.softmax(
        torch.tensor(retrieval_scores).to("cuda"), dim=0
    )
    marginalized = (
        retrieval_probs.unsqueeze(1) * label_logits
    ).sum(dim=0)
    return marginalized, output.loss


def evaluate(data, model, q_encoder, bart_tokenizer,
             q_tokenizer, index, passages, label2id):
    model.eval()
    q_encoder.eval()
    correct_3way = correct_2way = total_3way = total_2way = 0
    id2label = {v: k for k, v in label2id.items()}
    with torch.no_grad():
        for example in data:
            results, _ = search_index(
                example["claim"], index, passages,
                q_encoder, q_tokenizer,
                n_docs=config["model"]["n_docs"],
                training=False
            )
            marginalized, _ = forward_pass(
                example["claim"], results, example["label"],
                model, bart_tokenizer, label2id
            )
            pred_idx   = torch.argmax(marginalized).item()
            pred_label = id2label[str(pred_idx)]
            gold_label = example["label"]
            total_3way   += 1
            correct_3way += int(pred_label == gold_label)
            if gold_label != "NOT ENOUGH INFO":
                total_2way   += 1
                correct_2way += int(pred_label == gold_label)
    model.train()
    q_encoder.train()
    return correct_3way / total_3way, correct_2way / max(total_2way, 1)


print("All functions defined")

All functions defined


In [9]:
import time

example_test = list(dataset["train"])[0]
times = []
for _ in range(10):
    t0 = time.time()
    search_index(
        example_test["claim"], gpu_index, passages,
        q_encoder, q_tokenizer, training=False
    )
    times.append((time.time() - t0) * 1000)

ms = sum(times) / len(times)
print(f"GPU retrieval: {ms:.1f} ms per query")
print(f"Projected per epoch: {ms * 228277 / 1000 / 3600:.1f} hours")
print(f"(Original CPU was ~12 hours per epoch)")

GPU retrieval: 22.0 ms per query
Projected per epoch: 1.4 hours
(Original CPU was ~12 hours per epoch)


In [10]:
from torch.amp import GradScaler
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

def train(model, q_encoder, train_data, val_data,
          start_epoch=1, n_epochs=2):
    """
    Train for n_epochs starting from start_epoch.
    start_epoch=1 means we already did epoch 1,
    so this runs epochs 2 and 3.
    """
    GRAD_ACCUM  = config["training"]["gradient_accumulation_steps"]
    total_steps = (
        len(train_data) // config["training"]["batch_size"]
    ) * n_epochs

    optimizer = AdamW([
        {"params": q_encoder.parameters(),
         "lr": config["training"]["learning_rate"]},
        {"params": model.parameters(),
         "lr": config["training"]["learning_rate"]}
    ], weight_decay=config["training"]["weight_decay"])

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config["training"]["warmup_steps"],
        num_training_steps=total_steps
    )

    scaler  = GradScaler()
    loss_fn = torch.nn.CrossEntropyLoss(
        label_smoothing=config["training"]["label_smoothing"]
    )

    best_val_acc = 0.0

    for epoch in range(start_epoch, start_epoch + n_epochs):

        model.train()
        q_encoder.train()

        total_loss = correct = total = 0
        optimizer.zero_grad()

        for i, example in enumerate(train_data):

            retrieved_results, _ = search_index(
                example["claim"], gpu_index, passages,
                q_encoder, q_tokenizer,
                n_docs=config["model"]["n_docs"],
                training=True
            )

            marginalized, _ = forward_pass(
                example["claim"], retrieved_results,
                example["label"], model,
                bart_tokenizer, LABEL2ID
            )

            gold_idx = torch.tensor(
                [int(LABEL2ID[example["label"]])]
            ).to("cuda")
            loss = loss_fn(marginalized.unsqueeze(0), gold_idx)
            loss = loss / GRAD_ACCUM

            scaler.scale(loss).backward()

            if (i + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) +
                    list(q_encoder.parameters()),
                    config["training"]["max_grad_norm"]
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item() * GRAD_ACCUM
            pred        = torch.argmax(marginalized).item()
            correct    += int(pred == int(LABEL2ID[example["label"]]))
            total      += 1

            if i % 50 == 0:
                print(f"Epoch {epoch+1} | "
                      f"Step {i}/{len(train_data)} | "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Acc: {correct/max(total,1):.1%}")

        # handle remaining steps
        if len(train_data) % GRAD_ACCUM != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) +
                list(q_encoder.parameters()),
                config["training"]["max_grad_norm"]
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        # validation
        val_acc_3way, val_acc_2way = evaluate(
            val_data, model, q_encoder,
            bart_tokenizer, q_tokenizer,
            gpu_index, passages, LABEL2ID
        )

        print(f"\nEpoch {epoch+1} complete (GPU FAISS)")
        print(f"  Train acc:     {correct/total:.1%}")
        print(f"  Val 3-way acc: {val_acc_3way:.1%}  (target 72.5%)")
        print(f"  Val 2-way acc: {val_acc_2way:.1%}  (target 89.5%)")

        # save per-epoch checkpoint
        torch.save(
            q_encoder.state_dict(),
            os.path.join(CHECKPOINT_DIR,
                         f"q_encoder_epoch{epoch+1}_gpu.pt")
        )
        torch.save(
            model.state_dict(),
            os.path.join(CHECKPOINT_DIR,
                         f"bart_epoch{epoch+1}_gpu.pt")
        )

        # save best
        if val_acc_3way > best_val_acc:
            best_val_acc = val_acc_3way
            torch.save(
                q_encoder.state_dict(),
                os.path.join(CHECKPOINT_DIR, "q_encoder_best_gpu.pt")
            )
            torch.save(
                model.state_dict(),
                os.path.join(CHECKPOINT_DIR, "bart_best_gpu.pt")
            )
            print(f"  New best saved — acc: {best_val_acc:.1%}")

    print(f"\nTraining complete. Best val acc: {best_val_acc:.1%}")
    return best_val_acc


print("train() defined")

train() defined


In [11]:
# same seed as original notebook so data order is consistent
random.seed(config["data"]["seed"])
full_train = list(dataset["train"])
random.shuffle(full_train)
train_data = full_train

val_data_small = random.sample(
    list(dataset["validation"]), 500
)

print(f"Train: {len(train_data):,} | Val: {len(val_data_small):,}")

Train: 228,277 | Val: 500


In [16]:
# run epochs 2 and 3 from epoch 1 checkpoint
print("Starting epochs 2 and 3 with GPU FAISS...")
print("Expected: ~1.4 hours per epoch (~2.8 hours total)")
print("Checkpoints saved as: q_encoder_epoch2_gpu.pt, bart_epoch2_gpu.pt etc.\n")

best = train(
    model, q_encoder,
    train_data, val_data_small,
    start_epoch=2,   # already did epoch 1
    n_epochs=1       # run epochs 2 and 3
)

print(f"\nFinal best 3-way val accuracy: {best:.1%}")
print(f"Paper target: 72.5%")

Starting epochs 2 and 3 with GPU FAISS...
Expected: ~1.4 hours per epoch (~2.8 hours total)
Checkpoints saved as: q_encoder_epoch2_gpu.pt, bart_epoch2_gpu.pt etc.

Epoch 3 | Step 0/228277 | Loss: 0.2913 | Acc: 100.0%
Epoch 3 | Step 50/228277 | Loss: 0.4144 | Acc: 96.1%
Epoch 3 | Step 100/228277 | Loss: 0.5275 | Acc: 90.1%
Epoch 3 | Step 150/228277 | Loss: 0.5317 | Acc: 88.7%
Epoch 3 | Step 200/228277 | Loss: 0.5466 | Acc: 88.1%
Epoch 3 | Step 250/228277 | Loss: 0.5591 | Acc: 87.3%
Epoch 3 | Step 300/228277 | Loss: 0.5326 | Acc: 89.0%
Epoch 3 | Step 350/228277 | Loss: 0.5083 | Acc: 90.0%
Epoch 3 | Step 400/228277 | Loss: 0.5001 | Acc: 90.3%
Epoch 3 | Step 450/228277 | Loss: 0.5268 | Acc: 89.4%
Epoch 3 | Step 500/228277 | Loss: 0.5226 | Acc: 89.0%
Epoch 3 | Step 550/228277 | Loss: 0.5222 | Acc: 89.3%
Epoch 3 | Step 600/228277 | Loss: 0.5130 | Acc: 89.9%
Epoch 3 | Step 650/228277 | Loss: 0.5123 | Acc: 89.7%
Epoch 3 | Step 700/228277 | Loss: 0.5055 | Acc: 90.0%
Epoch 3 | Step 750/228277 | 